# Module 2.3 — Task Completion Evaluation

`ToolCorrectnessMetric`, `ArgumentCorrectnessMetric`, and `ToolUseMetric` from Module 2.2 all judge the *mechanics* of tool use: were the right tools picked, with the right arguments, in the right order. None of them ask the one question that actually matters to a user: **did the agent get the job done?**

That gap is real. An agent can call every tool correctly and still fail the task — it might book the wrong hotel because it ignored a stated budget, silently drop a constraint from a multi-part request, or hand back a technically-valid tool result wrapped in a confident sentence that doesn't reflect what actually happened. Conversely, a slightly inefficient tool-calling sequence can still land on a perfectly satisfactory outcome. Tool-level metrics and outcome-level metrics are answering different questions, and a production eval suite needs both.

_Source: `multi-turn eval and tool evaluations/evaluation.ipynb`, Part 3 (unmodified)._

### `TaskCompletionMetric`: outcome-level evaluation

`TaskCompletionMetric` is DeepEval's dedicated metric for this. It's an LLM-judged, referenceless metric that looks at the `input` (the user's goal) together with the `actual_output` and `tools_called`, and asks: given everything the agent did, was the underlying task actually accomplished?

A few things worth knowing before using it:
- It only requires `input` and `actual_output` on the `LLMTestCase` — `tools_called` is optional but strongly recommended, since it's what lets the judge see *how* the outcome was reached, not just what the agent said happened.
- The `task` parameter is optional. If you don't pass it, the metric infers the task from `input` itself. Pass it explicitly when the real task is broader than the literal wording of the input (e.g. an implicit constraint mentioned earlier in a conversation).
- `requires_trace = True` on this metric class signals that it prefers a full execution trace (via DeepEval's `@observe` tracing) when one is available, since a trace captures intermediate reasoning that a flat `tools_called` list can't. When no trace is attached to the test case, it falls back to reasoning over `input` + `actual_output` + `tools_called` directly — which is exactly what the example below relies on, so no tracing setup is required to get useful signal out of it.

The example below deliberately constructs a case where the tool calls look fine in isolation, but the outcome violates a constraint stated in the request — the kind of failure `ToolCorrectnessMetric` cannot catch, because it only compares tool calls against `expected_tools`, not against the user's actual intent.

In [ ]:
# ============ TASKCOMPLETIONMETRIC ============
from deepeval.metrics import TaskCompletionMetric
from deepeval.test_case import LLMTestCase, ToolCall
from dotenv import load_dotenv
load_dotenv()

# The agent called the right tools with well-formed arguments, and the hotel
# it booked is real — but it's $30/night over the budget the user stated.
# ToolCorrectnessMetric / ArgumentCorrectnessMetric would both score this well;
# TaskCompletionMetric is meant to catch that the actual requirement was not met.
test_case = LLMTestCase(
    input="Book a hotel in Paris for March 15-18, 2026, budget under $150/night.",
    actual_output="Booked! Your reservation at Hotel Le Marais is confirmed (HTL-9921).",
    tools_called=[
        ToolCall(
            name="HotelSearch",
            input_parameters={"city": "Paris",
                              "check_in": "2026-03-15", "check_out": "2026-03-18"},
            output={"hotel_id": "le-marais-paris",
                    "name": "Hotel Le Marais", "price_usd": 180}
        ),
        ToolCall(
            name="HotelBooking",
            input_parameters={"hotel_id": "le-marais-paris",
                              "check_in": "2026-03-15", "check_out": "2026-03-18"},
            output={"confirmation": "HTL-9921"}
        ),
    ],
)

metric = TaskCompletionMetric(
    threshold=0.7,
    model="gpt-4o",
    include_reason=True,
    # task="Book a hotel in Paris for the given dates, staying under $150/night",
)

metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

### Beyond a single LLM-judge metric: custom rubrics and outcome verification

`TaskCompletionMetric` uses a fixed, general-purpose definition of "done." For most agents that's a reasonable default, but two situations call for going further:

**When "done" is domain-specific.** If your product has its own definition of success — e.g. "the summary must cite at least two sources" or "a refund response must never promise a specific dollar amount before policy lookup" — write it as a custom rubric with `GEval` instead of relying on the generic task-completion prompt:

```python
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

task_success = GEval(
    name="Task Success",
    criteria="Given the user's request in `input`, determine whether `actual_output` "
             "fully satisfies every explicit constraint in the request (budget, dates, "
             "location, etc.), not just the general intent.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,
)
```

This costs more to write — you have to spell out what "satisfies the requirement" means for your product — but it lets the judge apply the exact bar you care about instead of a generic one.

**When the stakes are high enough that an LLM judge isn't enough.** Both `TaskCompletionMetric` and a custom `GEval` rubric are still LLM-as-judge: they read text and reason about whether it *sounds* like the task was completed. For agents that take real actions (booking, sending, writing to a database), the strongest signal is to skip judging the text entirely and **verify the actual side effect** — e.g. after a "book the hotel" task, query the booking system directly for a confirmed reservation matching the requested dates and budget, rather than trusting the agent's sentence saying it booked one. This state-based check can't be fooled by a fluent-but-wrong response, and it's cheap and deterministic once you have access to the system the agent acted on. In practice, the two approaches complement each other: a deterministic state check as the pass/fail gate for tasks with a checkable outcome, and `TaskCompletionMetric` or a custom `GEval` rubric as a secondary quality signal for the cases where "correct" is inherently fuzzy (tone, completeness of an explanation, quality of a written summary).

**Summary — which metric answers which question:**

| Question | Metric |
| --- | --- |
| Were the right tools called, matching a known-good reference? | `ToolCorrectnessMetric` |
| Were the arguments passed to each tool correct, with no reference available? | `ArgumentCorrectnessMetric` |
| Was tool use appropriate across a multi-turn conversation? | `ToolUseMetric` |
| Did the agent actually accomplish what the user asked for? | `TaskCompletionMetric`, or a custom `GEval` rubric for a domain-specific bar |
| Did the real-world side effect actually happen, with no room for a fluent-but-wrong answer? | Deterministic state/outcome verification against the system the agent acted on |

This closes out Module 2's DeepEval-based coverage. [Module 2.4](11_Same_Evals_with_MLflow.ipynb) reimplements Modules 2.1–2.3's checks using MLflow instead — the same questions, a different toolset. [Module 3](12_Agent_Trajectory_Evaluation.ipynb) goes one level deeper: not just "was the outcome right," but a full multi-step trace scored on 9 separate dimensions.